# HCDE: train, export, back up and evaluate the assistance-calibrated model

**Settings (right panel):** Accelerator **GPU T4** (x1 or x2, only one is used) · Internet **on**.

**Add-ons → Secrets**, add these and **tick each one** so this notebook can read it:

| Secret | Value |
| --- | --- |
| `HF_TOKEN` | Hugging Face token with **Write** access (automatic backup of every result) |
| `TEACHER_BASE_URL` | `https://api.groq.com/openai/v1` |
| `TEACHER_API_KEY` | your Groq key(s), comma-separated |
| `TEACHER_MODEL` | `openai/gpt-oss-120b` |
| `JUDGE_MODEL` | `qwen/qwen3.8-27b` |
| `JUDGE_EXTRA` | `{"reasoning_effort": "none"}` |
| `JUDGE_MAX_TOKENS` | `600` |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |

Then **Run All** and keep this tab open (Mac awake: `caffeinate -dims`).
Every result is uploaded to your private Hugging Face repo as soon as it exists, so nothing is lost if the session stops.
Each step skips itself when its output already exists (here or restored from Hugging Face), so after any crash just Run All again.

In [ ]:
import os, subprocess, glob, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU only: two make the Trainer split the model and run out of memory
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

BRANCH = "dev/dulanja"
# ---- choose the run (change only this line) ---------------------------------------------
EXPERIMENT = "qwen25-7b-3ep"      # "qwen25-7b-2ep" (run A) | "qwen25-7b-3ep" (run B) | "qwen3-8b" (run C)
# -------------------------------------------------------------------------------------------
CONFIGS = {
    "qwen25-7b-2ep": dict(base="Qwen/Qwen2.5-7B-Instruct", epochs=2, chatml=False),
    "qwen25-7b-3ep": dict(base="Qwen/Qwen2.5-7B-Instruct", epochs=3, chatml=False),  # keeps the best epoch
    "qwen3-8b":      dict(base="Qwen/Qwen3-8B",            epochs=3, chatml=True),   # plain ChatML, no thinking
}
CFG = CONFIGS[EXPERIMENT]
BASE_MODEL, EPOCHS = CFG["base"], CFG["epochs"]
CHATML = "--chatml" if CFG["chatml"] else ""
WORK = "/kaggle/working"
RUN = f"{WORK}/runs/{EXPERIMENT}"
GGUF = f"{WORK}/gguf-{EXPERIMENT}"
print("experiment:", EXPERIMENT, CFG)
REPO = f"{WORK}/AnchorAI"

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)

loaded = []
for k in ["HF_TOKEN", "TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
          "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS"]:
    if secret(k):
        os.environ[k] = secret(k)
        loaded.append(k)
print("secrets loaded:", loaded or "none")
if "HF_TOKEN" not in loaded:
    print("WARNING: no HF_TOKEN -> no automatic backup. Download files manually from the Output panel!")
if "TEACHER_BASE_URL" not in loaded:
    print("WARNING: no Groq secrets -> evaluation runs without the judge (no SDS scores).")
!git log --oneline -1

In [ ]:
# Backup helpers: a private Hugging Face repo "<your-username>/hcde-qwen7b".
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=os.environ.get("HF_TOKEN")) if os.environ.get("HF_TOKEN") else None
HF_REPO = None
if api:
    HF_REPO = f"{api.whoami()['name']}/hcde-{EXPERIMENT}"
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    print("backup repo:", f"https://huggingface.co/{HF_REPO}")

def backup(local_path, remote_path):
    """Upload a file or folder to the private HF repo. Never raises: a failed backup must not stop the run."""
    if not api or not os.path.exists(local_path):
        print("backup skipped:", local_path)
        return
    try:
        if os.path.isdir(local_path):
            api.upload_folder(repo_id=HF_REPO, folder_path=local_path, path_in_repo=remote_path)
        else:
            api.upload_file(repo_id=HF_REPO, path_or_fileobj=local_path, path_in_repo=remote_path)
        print("backed up:", local_path, "->", f"https://huggingface.co/{HF_REPO}/tree/main/{remote_path}")
    except Exception as e:
        print("BACKUP FAILED (download manually):", local_path, e)

def restore(remote_pattern, local_dir):
    """Bring earlier results back from the HF repo (used after a session restart)."""
    if not api:
        return
    try:
        snapshot_download(HF_REPO, repo_type="model", allow_patterns=[remote_pattern], local_dir=local_dir,
                          token=os.environ["HF_TOKEN"])
    except Exception as e:
        print("nothing to restore for", remote_pattern, f"({e.__class__.__name__})")

# After a restart: restore whatever an earlier session already produced, so finished steps are skipped.
restore("adapter/*", RUN); restore("training_meta.json", RUN)
restore("eval/*", RUN); restore("eval-transfer/*", RUN)
print("trained adapter available:", os.path.exists(f"{RUN}/adapter/adapter_config.json"))

In [ ]:
!pip install -q -U peft bitsandbytes accelerate
!pip uninstall -q -y torchao   # Kaggle's old torchao makes new peft refuse to load adapters (not needed here)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import transformers, peft, torch
print(transformers.__version__, peft.__version__, torch.__version__, "| GPUs visible:", torch.cuda.device_count())

## 1. Train (QLoRA)
2 epochs = 54 steps (~1 h 50 min), 3 epochs = 81 steps (~3 h 20 min). The best epoch on validation is kept.
The adapter is uploaded to Hugging Face as soon as training ends.

In [ ]:
# extra targeted training data (more SHARED examples), used when the file is in the repo
AUG = "components/c4_hcde/training/data/dataset/aug_shared.jsonl"
EXTRA = f"--extra-train {AUG}" if os.path.exists(AUG) else ""
print("extra training data:", AUG if EXTRA else "none")
if os.path.exists(f"{RUN}/adapter/adapter_config.json"):
    print("adapter already trained, skipping")
else:
    !python -m components.c4_hcde.training.train_qlora --base-model {BASE_MODEL} --out {RUN} --epochs {EPOCHS} {CHATML} {EXTRA}
    backup(f"{RUN}/adapter", "adapter")
    backup(f"{RUN}/training_meta.json", "training_meta.json")
!python -c "import json;m=json.load(open('{RUN}/training_meta.json'));print('best eval loss:', m['eval'].get('eval_loss'), '| minutes:', m['train_minutes'])"

## 2. Export to GGUF for Ollama on your Mac (about 20 min)
Merged model and temporary files go to `/tmp` (Kaggle's working folder only holds 20 GB).

In [ ]:
gguf_file = f"{GGUF}/hcde-q4_k_m.gguf"
if os.path.exists(gguf_file):
    print("GGUF already exported, skipping")
else:
    !apt-get -qq install -y cmake > /dev/null
    !python -m components.c4_hcde.training.export_gguf --base-model {BASE_MODEL} --adapter {RUN}/adapter --out {GGUF}
backup(gguf_file, "gguf/hcde-q4_k_m.gguf")
backup(f"{GGUF}/Modelfile", "gguf/Modelfile")
!ls -lh {GGUF}

## 3. Download links (backup if Hugging Face is not set up)
Click each link to download straight to your Mac.

In [ ]:
from IPython.display import FileLink, display
os.chdir(WORK)
!zip -qr hcde-{EXPERIMENT}.zip runs/{EXPERIMENT}
for f in [f"hcde-{EXPERIMENT}.zip", f"gguf-{EXPERIMENT}/Modelfile", f"gguf-{EXPERIMENT}/hcde-q4_k_m.gguf"]:
    if os.path.exists(f):
        display(FileLink(f))
os.chdir(REPO)

## 4. Quick look: one held-out task in every mode

In [ ]:
import json, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from components.c4_hcde.training.tiers import MODES, control_system_prompt

tok = AutoTokenizer.from_pretrained(f"{RUN}/adapter")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}), f"{RUN}/adapter")
task = json.loads(open("components/c4_hcde/training/data/dataset/test.jsonl").readline())["messages"][1]["content"]
print("TASK:", task, "\n")
for mode in MODES:
    msgs = [{"role": "system", "content": control_system_prompt(mode)}, {"role": "user", "content": task}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=400, do_sample=False)
    print(f"===== {mode} =====\n{tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)}\n")
del model; gc.collect(); torch.cuda.empty_cache()   # free the GPU for the evaluation process

## 5. Evaluate on the 16 held-out IT tasks (3 conditions, about 1 h + judge)
base model + short tag · base model + full rubric · fine-tuned model + short tag. Results are uploaded when done.
If the judge stops early (rate limits), the replies are already saved: rerun this cell and it continues.

In [ ]:
judge = "--judge" if os.environ.get("TEACHER_BASE_URL") else ""
if os.path.exists(f"{RUN}/eval/generations.jsonl"):
    # replies already generated (this session or restored): only (re)score what is missing
    !python -m components.c4_hcde.training.evaluate --from-generations {RUN}/eval/generations.jsonl --out {RUN}/eval {judge} --judge-conditions base_rubric tuned_tag
else:
    !python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter --out {RUN}/eval {judge} --judge-conditions base_rubric tuned_tag
backup(f"{RUN}/eval", "eval")

## 6. Transfer test: 20 non-IT tasks the model never saw (about 1.5 h + judge)

In [ ]:
T = "components/c4_hcde/training/data/dataset"
if os.path.exists(f"{RUN}/eval-transfer/generations.jsonl"):
    !python -m components.c4_hcde.training.evaluate --from-generations {RUN}/eval-transfer/generations.jsonl \
        --elements {T}/transfer_elements.json --out {RUN}/eval-transfer {judge} --judge-conditions base_rubric tuned_tag
else:
    !python -m components.c4_hcde.training.evaluate --base-model {BASE_MODEL} --adapter {RUN}/adapter \
        --conditions base_rubric tuned_tag --test {T}/transfer_raw.jsonl --elements {T}/transfer_elements.json --out {RUN}/eval-transfer {judge} --judge-conditions base_rubric tuned_tag
backup(f"{RUN}/eval-transfer", "eval-transfer")

## 7. Final check: everything that is backed up

In [ ]:
if api:
    for f in sorted(api.list_repo_files(HF_REPO)):
        print(f)
    print("\nOn your Mac:  huggingface-cli download", HF_REPO, "--local-dir ~/Desktop/HCDE/hcde-model")